# Module 04.11 — Alerting Rules (`alert`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.11 Alerting Rules (`alert`)

An alerting rule (saved as type `alert` — the UI renamed "alerts" to "rules" in Kibana 7.13
but the saved object type was never changed) periodically runs a **condition check** against
Elasticsearch and fires actions when that condition is met. The rule scheduler is driven by
the `schedule.interval` (or a cron expression) stored in the saved object.

The schema separates *what to check* from *where to send*:
- `params` contains the rule-type-specific query definition (e.g. ES|QL statement,
  threshold value, index pattern for an ES Query rule)
- `actions` contains an array of action definitions, each specifying a connector ID, a
  group (the trigger category), and message parameters
- `references` contains entries of type `action` pointing to the connector saved objects
  used in `actions`

Two things are **not** stored in the saved object:
- **Rule execution history** — past run results and alert states are stored in Elasticsearch
  system indices (`.kibana-event-log-*`, `.alerts-*`), not in the feature state. After a
  restore, rules restart with a clean execution history.
- **Rule scheduling state** — the rule will begin its schedule fresh from the restore point.

### Create in Kibana UI
1. Go to **Stack Management → Rules**
2. Click **Create rule**
3. Choose **Elasticsearch query** rule type
4. Configure: index=ecommerce, threshold: count > 1000 in 1 day
5. Add a server log action (connector)
6. Save as `Training — High Order Volume`

In [ ]:
heading('4.11 Alerting Rule — create via API')

rule_resp = kibana_post('/api/alerting/rule', {
    'name': 'Training — High Order Volume',
    'rule_type_id': '.es-query',
    'consumer': 'alerts',
    'schedule': {'interval': '1h'},
    'params': {
        'index': ['kibana_sample_data_ecommerce'],
        'timeField': 'order_date',
        'esQuery': json.dumps({'query': {'match_all': {}}}),
        'size': 100,
        'thresholdComparator': '>',
        'threshold': [500],
        'timeWindowSize': 24,
        'timeWindowUnit': 'h',
    },
    'actions': [],
    'notify_when': 'onActionGroupChange',
})
rule_id = rule_resp['id']
success(f'Rule created: id={rule_id}')

# Show the saved object schema
rules_so = find_saved_objects('alert')
training_rule = next((r for r in rules_so if r['id'] == rule_id), None)
if training_rule:
    pp(training_rule, 'alert saved object')
    info('Key fields in attributes:')
    info('  name            — display name')
    info('  rule_type_id    — identifies the rule type plugin (.es-query, etc.)')
    info('  schedule        — interval or cron expression')
    info('  params          — rule-type-specific parameters')
    info('  actions         — array of action objects (reference connectors via references[])')
    info('  notify_when     — onActiveAlert | onActionGroupChange | onThrottleInterval')
    info('  throttle        — minimum time between repeated alerts')
    info('  enabled         — whether the rule is active')
    info('references        — links to action (connector) saved objects')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link(
    '/app/management/insightsAndAlerting/triggersActions/rules',
    'Stack Management → Rules — view the rule we just created',
)
if 'rule_id' in dir():
    kibana_link(
        f'/app/management/insightsAndAlerting/triggersActions/rules/{rule_id}',
        f'Rule detail: Training — High Order Volume',
    )

In [ ]:
heading('Alerting Rule — snapshot → delete → restore')

# Ensure rule exists (re-runnable — rule_id set in cell above)
existing_rules = find_saved_objects('alert')
if not any(o['id'] == rule_id for o in existing_rules):
    warn('Rule missing — recreating')
    rule_resp = kibana_post('/api/alerting/rule', {
        'name': 'Training — High Order Value Alert',
        'rule_type_id': '.es-query',
        'consumer': 'alerts',
        'schedule': {'interval': '1h'},
        'params': {
            'index': ['kibana_sample_data_ecommerce'],
            'timeField': 'order_date',
            'esQuery': json.dumps({'query': {'range': {'taxful_total_price': {'gte': 200}}}}),
            'size': 100, 'thresholdComparator': '>', 'threshold': [0],
            'timeWindowSize': 1, 'timeWindowUnit': 'h',
            'excludeHitsFromPreviousRun': True,
        },
        'actions': [],
        'notify_when': 'onActionGroupChange',
    })
    rule_id = rule_resp['id']
    success(f'Rule recreated: id={rule_id}')

snap_delete_restore_cycle('snap-rule-test', 'Alerting Rule')

kibana_delete(f'/api/alerting/rule/{rule_id}')
warn(f'Accidentally deleted alerting rule {rule_id}')
assert not any(o['id'] == rule_id for o in find_saved_objects('alert')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-rule-test')
time.sleep(3)

restored = find_saved_objects('alert')
assert any(o['id'] == rule_id for o in restored), 'Rule should be restored'
success(f'Alerting rule {rule_id} successfully restored!')
kibana_link('/app/management/insightsAndAlerting/triggersActions/rules', 'Rules — verify restored rule')